# Fabric ETL portfolio assessment

Assess approved Alteryx, PowerCenter, and IDMC exports in Fabric without starting the web app. The notebook uses the same Python parsers and compiler-based target selection as the web app. It does not execute source workflows, save generated definitions, or deploy Fabric items.

Attach a default Lakehouse and a Fabric Runtime 2.0 environment with the accelerator wheel and approved dependencies. Stage only data approved for processing in OneLake. Run the configuration and preview cells first; then set `CONFIRM_ASSESSMENT = True` to assess the selected input.

In [ ]:
from pathlib import Path

LAKEHOUSE_FILES = Path('/lakehouse/default/Files')
INPUT_PATH = 'etl-assessment/input'  # File or directory relative to Lakehouse Files.
REPORT_FOLDER = 'etl-assessment/reports'
CONFIRM_ASSESSMENT = False
PERSIST_DELTA_TABLES = False
INITIALIZE_SCHEMA_ONLY = False
WRITE_REPORTS = False
REFRESH_SEMANTIC_MODEL = True
SEMANTIC_MODEL_NAME = 'ETL Migration Assessment Model'
SEMANTIC_MODEL_ID = ''


## Preview the input

Use a workflow file, a PowerCenter XML export, a supported archive, or a portfolio directory. To assess an extracted IDMC package, select its root containing `manifest.json` and `Explore`. When using a mixed estate directory, keep IDMC packages as ZIP files. Keep inputs unchanged during review and assessment.

In [ ]:
import csv
import json
import sys
from collections import Counter
from uuid import uuid4

if sys.version_info < (3, 13):
    raise RuntimeError('Attach a Fabric Spark environment with Python 3.13 or later.')

from etl_migration_core.graph import redact
from etl_target_fabric.portfolio import FabricAssessmentApplication as AssessmentApplication


def lakehouse_path(root, relative):
    if not root.is_dir():
        raise FileNotFoundError('Attach a default Lakehouse before continuing.')
    path = Path(relative)
    if path.anchor or not path.parts or '..' in path.parts:
        raise ValueError('Enter a nonempty path relative to Lakehouse Files, without parent segments.')
    result = root / path
    if not result.resolve().is_relative_to(root.resolve()):
        raise ValueError('The selected path is outside Lakehouse Files.')
    return result


def show_rows(rows, empty_message):
    if rows:
        display(spark.createDataFrame(rows))
    else:
        print(empty_message)


def input_signature(discovered):
    return tuple((item.display_path, item.sha256) for item in discovered.workflows)


assessment = None
preview_signature = ()
if INITIALIZE_SCHEMA_ONLY:
    print('Schema initialization mode: skipping input discovery.')
else:
    source = lakehouse_path(LAKEHOUSE_FILES, INPUT_PATH)
    application = AssessmentApplication()
    preview = application.preview(source)
    preview_signature = input_signature(preview)
    print(f'{preview.workflow_count} candidates; {preview.skipped_count} skipped.')
    show_rows([
        {'path': item.display_path, 'source': item.source_platform.value, 'bytes': item.size_bytes}
        for item in preview.workflows
    ], 'No supported assets discovered.')
    show_rows([
        {'path': redact(item.path), 'reason': item.reason.value}
        for item in preview.skipped
    ], 'Nothing skipped during discovery.')


## Assess and review portfolio

Confirm the reviewed selection before running this cell. A compiler-backed target is a recommendation, not proof of successful execution or semantic equivalence.

In [ ]:
assessment = None
if INITIALIZE_SCHEMA_ONLY:
    print('Schema initialization mode: skipping assessment execution.')
else:
    if not CONFIRM_ASSESSMENT:
        raise RuntimeError('Review the preview and set CONFIRM_ASSESSMENT = True before assessing.')
    if not preview_signature:
        raise ValueError('No supported input to assess.')
    if source != lakehouse_path(LAKEHOUSE_FILES, INPUT_PATH):
        raise ValueError('Input path changed. Run the preview again.')

    result = application.assess(source, route_alteryx=True)
    if input_signature(result) != preview_signature:
        raise RuntimeError('Input fingerprints changed after preview. Run the preview again.')
    assessment = result

    print(f'{assessment.workflow_count} assets assessed; {assessment.failed_count} inputs failed.')
    for field in ('source_platform', 'complexity_band', 'recommended_target', 'migration_wave'):
        counts = Counter(getattr(item, field) for item in assessment.assessments)
        show_rows([
            {'dimension': field, 'value': str(value), 'assets': count}
            for value, count in sorted(counts.items())
        ], f'No results for {field}.')
    show_rows([{'failure': redact(message)} for message in assessment.failures], 'No assessment failures.')


## Persist normalized assessment tables

Set `PERSIST_DELTA_TABLES = True`, rerun the configuration cell, and run this cell to append an immutable assessment run to eight Delta tables. The current run becomes the latest completed run only after all child tables are written. Values are bounded and redacted; effort estimates, absolute input paths, and raw expressions aren't persisted.

In [ ]:
if INITIALIZE_SCHEMA_ONLY or PERSIST_DELTA_TABLES:
    from pyspark.sql.types import BooleanType, DoubleType, IntegerType, StringType, StructField, StructType, TimestampType

    schemas = {
        'etl_assessment_run': StructType([
            StructField('RunId', StringType(), False), StructField('RunLabel', StringType(), False),
            StructField('StartedAtUtc', TimestampType(), False), StructField('CompletedAtUtc', TimestampType(), False),
            StructField('Status', StringType(), False), StructField('WorkflowCount', IntegerType(), False),
            StructField('FailedCount', IntegerType(), False), StructField('SkippedCount', IntegerType(), False),
            StructField('RulesVersion', StringType(), False), StructField('IsLatestCompletedRun', BooleanType(), False),
        ]),
        'etl_workflow_assessment': StructType([
            StructField('WorkflowKey', StringType(), False), StructField('RunId', StringType(), False),
            StructField('WorkflowPath', StringType(), False), StructField('SourcePlatform', StringType(), False),
            StructField('AssetType', StringType(), False), StructField('NativeId', StringType(), True),
            StructField('Folder', StringType(), True), StructField('ToolCount', IntegerType(), False),
            StructField('ConnectionCount', IntegerType(), False), StructField('ComplexityScore', IntegerType(), False),
            StructField('ComplexityBand', StringType(), False), StructField('RecommendedTarget', StringType(), False),
            StructField('MigrationWave', StringType(), False), StructField('RecommendationReason', StringType(), False),
            StructField('RegistryVersion', StringType(), False),
        ]),
        'etl_tool_inventory': StructType([
            StructField('RunId', StringType(), False), StructField('WorkflowKey', StringType(), False),
            StructField('ToolName', StringType(), False), StructField('ToolCount', IntegerType(), False),
            StructField('SupportCategory', StringType(), False),
        ]),
        'etl_finding': StructType([
            StructField('RunId', StringType(), False), StructField('WorkflowKey', StringType(), False),
            StructField('FindingType', StringType(), False), StructField('FindingText', StringType(), False),
        ]),
        'etl_dependency': StructType([
            StructField('RunId', StringType(), False), StructField('WorkflowKey', StringType(), False),
            StructField('DependencyReference', StringType(), False),
        ]),
        'etl_workflow_node': StructType([
            StructField('RunId', StringType(), False), StructField('WorkflowKey', StringType(), False),
            StructField('NodeId', StringType(), False), StructField('NodeName', StringType(), False),
            StructField('NodeLabel', StringType(), False), StructField('Plugin', StringType(), False),
            StructField('SupportCategory', StringType(), False), StructField('Disabled', BooleanType(), False),
            StructField('X', DoubleType(), True), StructField('Y', DoubleType(), True),
        ]),
        'etl_workflow_connection': StructType([
            StructField('RunId', StringType(), False), StructField('WorkflowKey', StringType(), False),
            StructField('SourceNodeId', StringType(), False), StructField('TargetNodeId', StringType(), False),
            StructField('SourceAnchor', StringType(), False), StructField('TargetAnchor', StringType(), False),
        ]),
        'etl_capacity_profile': StructType([
            StructField('RunId', StringType(), False), StructField('WorkflowKey', StringType(), False),
            StructField('ProfileKind', StringType(), False), StructField('TargetComponents', StringType(), False),
            StructField('ComplexityUnits', DoubleType(), False), StructField('DestCount', IntegerType(), False),
            StructField('HeavySteps', IntegerType(), False), StructField('NotebookSteps', IntegerType(), False),
            StructField('PipelineSteps', IntegerType(), False), StructField('TotalSteps', IntegerType(), False),
            StructField('CoveredByLinkedMapping', BooleanType(), False),
        ]),
    }
    for table_name, schema in schemas.items():
        spark.createDataFrame([], schema).write.format('delta').mode('ignore').saveAsTable(table_name)

if INITIALIZE_SCHEMA_ONLY and not PERSIST_DELTA_TABLES:
    print('Initialized 8 empty ETL assessment Delta tables in the default Lakehouse.')
elif PERSIST_DELTA_TABLES:
    from datetime import datetime, timezone
    from hashlib import sha256
    from uuid import uuid4

    from etl_migration_core.discovery import DiscoveryService
    from etl_migration_core.ir import SourcePlatform
    from etl_migration_core.sources.defaults import create_default_registry
    from etl_portfolio_assessment.evidence import etl_asset_evidence as _base_etl_evidence, workflow_evidence as _base_wf_evidence
    from etl_target_fabric.capacity import COUNT_KEYS, INFORMATICA_COUNT_KEYS, INFORMATICA_WEIGHTS, WEIGHTS, build_capacity_profile
    from etl_target_fabric.support_matrix import alteryx_support_for, etl_support_for

    def workflow_evidence(staged_path, display_path, idx):
        return _base_wf_evidence(staged_path, display_path, idx, support_classifier=alteryx_support_for)

    def etl_asset_evidence(pkg, asset, display_path, idx):
        return _base_etl_evidence(pkg, asset, display_path, idx, support_classifier=etl_support_for)

    if assessment is None or not assessment.assessments:
        raise ValueError('Complete a nonempty assessment before persisting Delta tables.')
    if source != lakehouse_path(LAKEHOUSE_FILES, INPUT_PATH):
        raise ValueError('Input path changed. Preview and assess the new input before persisting.')

    discovery = DiscoveryService()
    registry = create_default_registry()
    discovered = discovery.discover(source)
    evidence_items = []
    for index, candidate in enumerate(discovered.workflows):
        with discovery.staged_workflow(candidate) as staged:
            if candidate.source_platform is SourcePlatform.ALTERYX:
                evidence_items.append(workflow_evidence(staged, candidate.display_path, index))
            else:
                detected = registry.detect(staged, candidate.display_path)
                package = registry.parse(staged, detected.match)
                evidence_items.extend(
                    etl_asset_evidence(package, asset, candidate.display_path, index)
                    for asset in package.assets
                )

    run_id = str(uuid4())
    completed_at = datetime.now(timezone.utc)
    run_label = completed_at.strftime('%Y-%m-%d %H:%M UTC')
    workflow_keys = {
        item.path: sha256(f'{run_id}|{item.path}'.encode()).hexdigest()
        for item in assessment.assessments
    }
    evidence_by_path = {item.workflow_path: item for item in evidence_items}
    candidates_by_path = {item.display_path: item for item in discovered.workflows}

    run_rows = [{
        'RunId': run_id,
        'RunLabel': run_label,
        'StartedAtUtc': completed_at,
        'CompletedAtUtc': completed_at,
        'Status': 'completed',
        'WorkflowCount': assessment.workflow_count,
        'FailedCount': assessment.failed_count,
        'SkippedCount': len(assessment.skipped),
        'RulesVersion': ','.join(sorted({item.rules_version for item in assessment.assessments})),
        'IsLatestCompletedRun': False,
    }]
    workflow_rows = [
        {
            'WorkflowKey': workflow_keys[item.path], 'RunId': run_id, 'WorkflowPath': item.path,
            'SourcePlatform': item.source_platform.value, 'AssetType': item.asset_type.value,
            'NativeId': item.native_id, 'Folder': item.folder, 'ToolCount': item.tool_count,
            'ConnectionCount': item.connection_count, 'ComplexityScore': item.complexity_score,
            'ComplexityBand': item.complexity_band, 'RecommendedTarget': item.recommended_target,
            'MigrationWave': item.migration_wave, 'RecommendationReason': redact(item.recommendation_reason)[:2000],
            'RegistryVersion': item.registry_version,
        }
        for item in assessment.assessments
    ]
    tool_rows = [
        {'RunId': run_id, 'WorkflowKey': workflow_keys[item.path], 'ToolName': redact(tool.name)[:256], 'ToolCount': tool.count, 'SupportCategory': tool.support}
        for item in assessment.assessments for tool in item.tools
    ]
    finding_rows = []
    dependency_rows = []
    node_rows = []
    connection_rows = []
    capacity_rows = []
    for item in assessment.assessments:
        key = workflow_keys[item.path]
        profile = item.capacity_profile
        if profile is None and item.source_platform.value == 'alteryx' and item.path in candidates_by_path:
            with discovery.staged_workflow(candidates_by_path[item.path]) as wf_path:
                profile = build_capacity_profile(wf_path)
        if profile is not None:
            active_weights = INFORMATICA_WEIGHTS if profile.profile_kind == 'informatica' else WEIGHTS
            active_keys = INFORMATICA_COUNT_KEYS if profile.profile_kind == 'informatica' else COUNT_KEYS
            capacity_rows.append({
                'RunId': run_id,
                'WorkflowKey': key,
                'ProfileKind': profile.profile_kind,
                'TargetComponents': ', '.join(item.target_components),
                'ComplexityUnits': float(sum(profile.counts[k] * active_weights[k] for k in active_keys)),
                'DestCount': int(profile.counts.get('dest', 0)),
                'HeavySteps': int(profile.heavy_steps),
                'NotebookSteps': int(profile.notebook_steps),
                'PipelineSteps': int(profile.pipeline_steps),
                'TotalSteps': int(profile.total_steps),
                'CoveredByLinkedMapping': bool(profile.covered_by_linked_mapping),
            })
        finding_rows.append({'RunId': run_id, 'WorkflowKey': key, 'FindingType': 'Recommendation', 'FindingText': redact(item.recommendation_reason)[:2000]})
        finding_rows.extend(
            {'RunId': run_id, 'WorkflowKey': key, 'FindingType': 'UnsupportedTool', 'FindingText': redact(value)[:2000]}
            for value in item.unsupported_tools
        )
        dependency_rows.extend(
            {'RunId': run_id, 'WorkflowKey': key, 'DependencyReference': redact(value)[:2000]}
            for value in item.dependencies
        )
        evidence = evidence_by_path.get(item.path)
        if evidence is None:
            continue
        finding_rows.extend(
            {'RunId': run_id, 'WorkflowKey': key, 'FindingType': 'Parser', 'FindingText': redact(value)[:2000]}
            for value in evidence.findings
        )
        node_rows.extend(
            {
                'RunId': run_id, 'WorkflowKey': key, 'NodeId': node.id, 'NodeName': redact(node.name)[:256],
                'NodeLabel': redact(node.label)[:256], 'Plugin': redact(node.plugin)[:512],
                'SupportCategory': node.support, 'Disabled': node.disabled, 'X': node.x, 'Y': node.y,
            }
            for node in evidence.nodes
        )
        connection_rows.extend(
            {
                'RunId': run_id, 'WorkflowKey': key, 'SourceNodeId': edge.source, 'TargetNodeId': edge.target,
                'SourceAnchor': edge.source_anchor, 'TargetAnchor': edge.target_anchor,
            }
            for edge in evidence.edges
        )

    records = {
        'etl_workflow_assessment': workflow_rows, 'etl_tool_inventory': tool_rows,
        'etl_finding': finding_rows, 'etl_dependency': dependency_rows,
        'etl_workflow_node': node_rows, 'etl_workflow_connection': connection_rows,
        'etl_capacity_profile': capacity_rows,
    }
    written = []
    try:
        for table_name, rows in records.items():
            spark.createDataFrame(rows, schemas[table_name]).write.format('delta').mode('append').saveAsTable(table_name)
            written.append(table_name)
        spark.createDataFrame(run_rows, schemas['etl_assessment_run']).write.format('delta').mode('append').saveAsTable('etl_assessment_run')
        written.append('etl_assessment_run')
        spark.sql(
            "UPDATE etl_assessment_run SET IsLatestCompletedRun = "
            f"CASE WHEN RunId = '{run_id}' THEN true ELSE false END WHERE Status = 'completed'"
        )
    except Exception:
        for table_name in reversed(written):
            spark.sql(f"DELETE FROM {table_name} WHERE RunId = '{run_id}'")
        raise
    print(f'Persisted run {run_id} to 8 Delta tables; marked it as the latest completed run.')
    if REFRESH_SEMANTIC_MODEL:
        def _refresh_direct_lake_model():
            import json as _json
            import time as _time
            import urllib.request as _req

            nb_utils = globals().get('notebookutils') or globals().get('mssparkutils')
            if nb_utils is None:
                return
            ws_id = ''
            try:
                import sempy.fabric as _sempy_fabric
                ws_id = str(_sempy_fabric.get_workspace_id())
            except Exception:
                return
            if not ws_id:
                return
            fabric_token = nb_utils.credentials.getToken('https://api.fabric.microsoft.com')
            model_id = SEMANTIC_MODEL_ID.strip()
            if not model_id:
                list_req = _req.Request(
                    f'https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/semanticModels',
                    headers={'Authorization': f'Bearer {fabric_token}', 'x-ms-fabric-skill': 'semantic-model-authoring'},
                )
                with _req.urlopen(list_req) as resp:
                    items = _json.loads(resp.read().decode('utf-8')).get('value', [])
                matches = [m for m in items if m.get('displayName') == SEMANTIC_MODEL_NAME]
                if len(matches) != 1:
                    print(f'Skipping semantic model refresh: found {len(matches)} models named {SEMANTIC_MODEL_NAME!r}.')
                    return
                model_id = matches[0]['id']
            pbi_token = nb_utils.credentials.getToken('https://analysis.windows.net/powerbi/api')
            refresh_req = _req.Request(
                f'https://api.powerbi.com/v1.0/myorg/groups/{ws_id}/datasets/{model_id}/refreshes',
                data=_json.dumps({'notifyOption': 'NoNotification'}).encode('utf-8'),
                headers={
                    'Authorization': f'Bearer {pbi_token}',
                    'Content-Type': 'application/json',
                    'x-ms-fabric-skill': 'semantic-model-authoring',
                },
                method='POST',
            )
            with _req.urlopen(refresh_req):
                pass
            for _ in range(30):
                _time.sleep(2)
                poll_req = _req.Request(
                    f'https://api.powerbi.com/v1.0/myorg/groups/{ws_id}/datasets/{model_id}/refreshes?$top=1',
                    headers={'Authorization': f'Bearer {pbi_token}', 'x-ms-fabric-skill': 'semantic-model-authoring'},
                )
                with _req.urlopen(poll_req) as resp:
                    latest = (_json.loads(resp.read().decode('utf-8')).get('value') or [None])[0]
                if not latest:
                    continue
                if latest.get('status') == 'Completed':
                    print(f'Refreshed Direct Lake semantic model {model_id}.')
                    return
                if latest.get('status') in ('Failed', 'Cancelled', 'Disabled'):
                    print(f'Warning: semantic model refresh ended in {latest.get("status")}: {latest.get("serviceExceptionJson")}')
                    return

        try:
            _refresh_direct_lake_model()
        except Exception as exc:
            print(f'Warning: semantic model refresh skipped ({exc}).')
else:
    print('Delta persistence is off. Set PERSIST_DELTA_TABLES = True to write normalized tables.')


## Export reports

Set `WRITE_REPORTS = True` and rerun this cell to save a new CSV and JSON report to the Lakehouse. Reports omit estimated effort, source configuration, and absolute input paths. File names, asset identifiers, and hashes remain in the JSON so the offline HTML can match supported inputs.

In [ ]:
def workflow_record(item):
    return {
        'path': item.path,
        'source_platform': item.source_platform.value,
        'asset_type': item.asset_type.value,
        'tool_count': item.tool_count,
        'connection_count': item.connection_count,
        'complexity_score': item.complexity_score,
        'complexity_band': item.complexity_band,
        'recommended_target': item.recommended_target,
        'migration_wave': item.migration_wave,
    }


def csv_value(value):
    if isinstance(value, str) and (value.startswith(('\t', '\r', '\n')) or value.lstrip().startswith(('=', '+', '-', '@'))):
        return "'" + value
    return value


if WRITE_REPORTS and not INITIALIZE_SCHEMA_ONLY:
    if assessment is None or not assessment.assessments:
        raise ValueError('Complete a nonempty assessment before exporting.')
    if source != lakehouse_path(LAKEHOUSE_FILES, INPUT_PATH):
        raise ValueError('Input path changed. Preview and assess the new input before exporting.')
    output_root = lakehouse_path(LAKEHOUSE_FILES, REPORT_FOLDER)
    if output_root.resolve().is_relative_to(source.resolve()):
        raise ValueError('Choose a report folder outside the input path.')
    output = output_root / f'run-{uuid4().hex}'
    output.mkdir(parents=True, exist_ok=False)
    ordered = sorted(assessment.assessments, key=lambda item: (-item.complexity_score, item.path))
    rows = [workflow_record(item) for item in ordered]
    report = {
        'workflow_count': assessment.workflow_count,
        'failed_count': assessment.failed_count,
        'workflows': [item.model_dump(mode='json', include={'display_path', 'sha256'}) for item in assessment.workflows],
        'assessments': [
            {**row, **item.model_dump(mode='json', include={'native_id', 'folder', 'rules_version', 'registry_version'})}
            for row, item in zip(rows, ordered, strict=True)
        ],
    }
    with (output / 'assessment.csv').open('x', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows({key: csv_value(value) for key, value in row.items()} for row in rows)
    with (output / 'assessment.json').open('x', encoding='utf-8') as stream:
        json.dump(report, stream, indent=2)
    print(f'Saved assessment.csv and assessment.json in Files/{output.relative_to(LAKEHOUSE_FILES).as_posix()}')
else:
    print('Report writing is off. Set WRITE_REPORTS = True to export.')
